<a href="https://colab.research.google.com/github/thorayya/online_exam_cheating_detection/blob/main/eye_tracking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install mediapipe
!pip install one-euro-filter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 7.7 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
ERROR: Could not find a version that satisfies the requirement one-euro-filter (from versions: none)
ERROR: No matching distribution found for one-euro-filter


In [ ]:
!wget -O face_landmarker_v2_with_blendshapes.task -q https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task

In [ ]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import time
from google.colab import files
from google.colab.patches import cv2_imshow
import statistics


In [ ]:
uploaded = files.upload()

Saving video.mp4 to video.mp4


In [ ]:
cap = cv2.VideoCapture("video.mp4")

base_options = python.BaseOptions(model_asset_path='face_landmarker_v2_with_blendshapes.task')
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    output_face_blendshapes=True,
    output_facial_transformation_matrixes=True,
    num_faces=1,
    running_mode=vision.RunningMode.VIDEO)

detector = vision.FaceLandmarker.create_from_options(options)

fps = cap.get(cv2.CAP_PROP_FPS)
frame_id = 0

def left_eye(landmark):
  left_eye_points = [ landmark[i] for i in LEFT_EYE ]

  eye_left = min(p.x for p in left_eye_points)
  eye_right = max(p.x for p in left_eye_points)
  eye_top = min(p.y for p in left_eye_points)
  eye_bottom = max(p.y for p in left_eye_points)

  return (eye_left, eye_right, eye_top, eye_bottom)

  # left_eye_pixels = [
  #     (int(p.x * frame.shape[1]), int(p.y * frame.shape[0]))
  #     for p in left_eye_points
  #   ]

  # for (x, y) in left_eye_pixels:
  #   cv2.circle(frame, (x, y), 2, (0, 255, 0), -1)


def right_eye(landmark):

  right_eye_points = [ landmark[i] for i in RIGHT_EYE]

  eye_left = min(p.x for p in right_eye_points)
  eye_right = max(p.x for p in right_eye_points)
  eye_top = min(p.y for p in right_eye_points)
  eye_bottom = max(p.y for p in right_eye_points)

  return (eye_left, eye_right, eye_top, eye_bottom)


def mean_normalized(landmark):

  left_iris_points = [ landmark[i] for i in LEFT_IRIS ]
  right_iris_points = [ landmark[i] for i in RIGHT_IRIS ]


  mean_left_iris = (sum(p.x for p in left_iris_points)/len(left_iris_points),
   sum(p.y for p in left_iris_points)/len(left_iris_points))

  mean_right_iris = (sum(p.x for p in right_iris_points)/len(right_iris_points),
    sum(p.y for p in right_iris_points)/len(right_iris_points))


  #just for test
  print(mean_right_iris, mean_left_iris)

  return [mean_left_iris, mean_right_iris]




LEFT_EYE =[ 362, 382, 381, 380, 374, 373, 390, 249, 263, 466, 388, 387, 386, 385,384, 398 ]
RIGHT_EYE=[ 33, 7, 163, 144, 145, 153, 154, 155, 133, 173, 157, 158, 159, 160, 161 , 246 ]
LEFT_IRIS = [468, 469, 470, 471, 472]
RIGHT_IRIS = [473, 474, 475, 476, 477]


calibrated = False
calibration_samples = []
start_time = time.time()


while cap.isOpened():
  ret, frame = cap.read()

  if not ret:
    print("not found")
    break

  frame = cv2.flip(frame, 1)
  frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


  mp_image = mp.Image(
  image_format = mp.ImageFormat.SRGB,
  data = frame)

  timestamp_ms = int((frame_id / fps) * 1000)
  detection_result = detector.detect_for_video(mp_image, timestamp_ms)

  # Process the detection result.
  if detection_result.face_landmarks:

    landmark = detection_result.face_landmarks[0]

    iris_center = mean_normalized(landmark)

    # --------------calibration-------------

    if not calibrated:

      calibration_samples.append([iris_center[0], iris_center[1]])

      if time.time() - start_time >= 3:

        baseline_left = (statistics.mean([sample[0][0] for sample in calibration_samples]),
                         statistics.mean([sample[0][1] for sample in calibration_samples]))

        baseline_right = (statistics.mean([sample[1][0] for sample in calibration_samples]),
                          statistics.mean([sample[1][1] for sample in calibration_samples]))

        calibrated = True
        print("Calibration done ✔")


    else:

      eye_right_box = right_eye(landmark)
      eye_left_box = left_eye(landmark)

      gaze_right_x = (iris_center[0][0] - eye_right_box[0])/(eye_right_box[1] - eye_right_box[0])
      gaze_right_y = (iris_center[0][1] - eye_right_box[2])/(eye_right_box[3] - eye_right_box[2])

      gaze_left_x = (iris_center[1][0] - eye_left_box[0])/(eye_left_box[1] - eye_left_box[0])
      gaze_left_y = (iris_center[1][1] - eye_left_box[2])/(eye_left_box[3] - eye_left_box[2])


  # print(f"gaze_right_x: {gaze_right_x}, gaze_right_y: {gaze_right_y}")
  # print(f"gaze_left_x: {gaze_left_x}, gaze_left_y: {gaze_left_y}")



  if cv2.waitKey(1) & 0xFF == ord('q'):
    break

  frame_id += 1

cap.release()
cv2.destroyAllWindows()


(0.5518757104873657, 0.3932750642299652) (0.43516346216201784, 0.37809500098228455)
0.5518757104873657
(0.552189314365387, 0.3934246122837067) (0.43538888096809386, 0.37809093594551085)
0.5520325124263763
(0.5537362337112427, 0.3944792330265045) (0.43715832233428953, 0.37826749086380007)
0.5526004195213318
(0.5545035123825073, 0.395504766702652) (0.43808685541152953, 0.37867064476013185)
0.5530761927366257
(0.5547509312629699, 0.39633622765541077) (0.4384196996688843, 0.37903430461883547)
0.5534111404418945
(0.5548143744468689, 0.3973215341567993) (0.43862211108207705, 0.379456502199173)
0.5536450127760569
(0.5547737717628479, 0.3982940077781677) (0.43865976929664613, 0.3799574077129364)
0.5538062640598842
(0.5547644853591919, 0.3990743100643158) (0.4386420965194702, 0.3804100751876831)
0.5539260417222976
(0.5547178983688354, 0.40037217140197756) (0.4386875629425049, 0.380881142616272)
0.5540140257941352
(0.5546899557113647, 0.4018098771572113) (0.4387289583683014, 0.3816088378429413)
